In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

# Spark SQL Consultas e Seleções

In [3]:
df = spark.sql('''select 'spark' as hello''')
df.show()

+-----+
|hello|
+-----+
|spark|
+-----+



# Importing DATA

In [4]:
df = spark.read.csv('/content/cereal.csv', sep = ',', inferSchema=True, header=True)

In [5]:
df.show(5)

+--------------------+---+----+--------+-------+---+------+-----+-----+------+------+--------+-----+------+----+---------+
|                name|mfr|type|calories|protein|fat|sodium|fiber|carbo|sugars|potass|vitamins|shelf|weight|cups|   rating|
+--------------------+---+----+--------+-------+---+------+-----+-----+------+------+--------+-----+------+----+---------+
|           100% Bran|  N|   C|      70|      4|  1|   130| 10.0|  5.0|     6|   280|      25|    3|   1.0|0.33|68.402973|
|   100% Natural Bran|  Q|   C|     120|      3|  5|    15|  2.0|  8.0|     8|   135|       0|    3|   1.0| 1.0|33.983679|
|            All-Bran|  K|   C|      70|      4|  1|   260|  9.0|  7.0|     5|   320|      25|    3|   1.0|0.33|59.425505|
|All-Bran with Ext...|  K|   C|      50|      4|  0|   140| 14.0|  8.0|     0|   330|      25|    3|   1.0| 0.5|93.704912|
|      Almond Delight|  R|   C|     110|      2|  2|   200|  1.0| 14.0|     8|    -1|      25|    3|   1.0|0.75|34.384843|
+---------------

# Manipulation Data With Spark SQL


In [6]:
df.createOrReplaceTempView("cereal")

In [8]:
cereal = spark.sql('''select * from cereal where mfr ='G' ''')
cereal.show(5)

+--------------------+---+----+--------+-------+---+------+-----+-----+------+------+--------+-----+------+----+---------+
|                name|mfr|type|calories|protein|fat|sodium|fiber|carbo|sugars|potass|vitamins|shelf|weight|cups|   rating|
+--------------------+---+----+--------+-------+---+------+-----+-----+------+------+--------+-----+------+----+---------+
|Apple Cinnamon Ch...|  G|   C|     110|      2|  2|   180|  1.5| 10.5|    10|    70|      25|    1|   1.0|0.75|29.509541|
|             Basic 4|  G|   C|     130|      3|  2|   210|  2.0| 18.0|     8|   100|      25|    3|  1.33|0.75|37.038562|
|            Cheerios|  G|   C|     110|      6|  2|   290|  2.0| 17.0|     1|   105|      25|    1|   1.0|1.25|50.764999|
|Cinnamon Toast Cr...|  G|   C|     120|      1|  3|   210|  0.0| 13.0|     9|    45|      25|    2|   1.0|0.75|19.823573|
|            Clusters|  G|   C|     110|      3|  2|   140|  2.0| 13.0|     7|   105|      25|    3|   1.0| 0.5|40.400208|
+---------------

# SELECT DISTINCT

In [10]:
cereal = spark.sql(''' select distinct name, type, mfr from cereal ''')
cereal.show()

+--------------------+----+---+
|                name|type|mfr|
+--------------------+----+---+
| Frosted Mini-Wheats|   C|  K|
|Just Right Crunch...|   C|  K|
|Just Right Fruit ...|   C|  K|
|       Count Chocula|   C|  G|
|             Crispix|   C|  K|
|      Fruity Pebbles|   C|  P|
|Muesli Raisins; P...|   C|  R|
|           Special K|   C|  K|
|Cinnamon Toast Cr...|   C|  G|
|Mueslix Crispy Blend|   C|  K|
|        Puffed Wheat|   C|  Q|
|         Raisin Bran|   C|  K|
|          Wheat Chex|   C|  R|
|           100% Bran|   C|  N|
|       Rice Krispies|   C|  K|
|     Raisin Nut Bran|   C|  G|
|      Frosted Flakes|   C|  K|
|        Lucky Charms|   C|  G|
|         Bran Flakes|   C|  P|
|    Honey Graham Ohs|   C|  Q|
+--------------------+----+---+
only showing top 20 rows



# WHERE no Spark SQL

In [11]:
cereal = spark.sql(''' select distinct type, mfr from cereal where mfr = 'K' ''')
cereal.show()

+----+---+
|type|mfr|
+----+---+
|   C|  K|
+----+---+



# GROUP BY

In [12]:
cereal = spark.sql(''' select type, mfr from cereal where mfr = 'K' group by 1,2 ''')
cereal.show()

+----+---+
|type|mfr|
+----+---+
|   C|  K|
+----+---+



# CASE WHEN

In [14]:
cereal = spark.sql(''' select
                            case when mfr = 'N' then 'teste N'
                                  when mfr = 'G' then 'teste G'
                            else 'RESTO' end as type,
                            mfr
                        from
                          cereal ''')
cereal.show()

+-------+---+
|   type|mfr|
+-------+---+
|teste N|  N|
|  RESTO|  Q|
|  RESTO|  K|
|  RESTO|  K|
|  RESTO|  R|
|teste G|  G|
|  RESTO|  K|
|teste G|  G|
|  RESTO|  R|
|  RESTO|  P|
|  RESTO|  Q|
|teste G|  G|
|teste G|  G|
|teste G|  G|
|teste G|  G|
|  RESTO|  R|
|  RESTO|  K|
|  RESTO|  K|
|teste G|  G|
|  RESTO|  K|
+-------+---+
only showing top 20 rows



# MIN/ MAX/ AVG/ COUNT

In [15]:
cereal = spark.sql(''' select
                            mfr,
                            type,
                            sum(calories),
                            min(calories),
                            max(calories),
                            avg(calories),
                            count(distinct name),
                            count(name)
                        from
                          cereal
                        group by
                          mfr, type
                        order by 1 ''')
cereal.show(5)

+---+----+-------------+-------------+-------------+------------------+--------------------+-----------+
|mfr|type|sum(calories)|min(calories)|max(calories)|     avg(calories)|count(DISTINCT name)|count(name)|
+---+----+-------------+-------------+-------------+------------------+--------------------+-----------+
|  A|   H|          100|          100|          100|             100.0|                   1|          1|
|  G|   C|         2450|          100|          140|111.36363636363636|                  22|         22|
|  K|   C|         2500|           50|          160|108.69565217391305|                  23|         23|
|  N|   H|          100|          100|          100|             100.0|                   1|          1|
|  N|   C|          420|           70|           90|              84.0|                   5|          5|
+---+----+-------------+-------------+-------------+------------------+--------------------+-----------+
only showing top 5 rows



# JOIN

In [16]:
sales = spark.read.csv('/content/sales_data_sample.csv', sep = ',', inferSchema=True, header=True)

In [17]:
sales.show(5)

+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+-----+----------+-------+---------+---------------+----------------+--------+
|ORDERNUMBER|QUANTITYORDERED|PRICEEACH|ORDERLINENUMBER|  SALES|      ORDERDATE| STATUS|QTR_ID|MONTH_ID|YEAR_ID|PRODUCTLINE|MSRP|PRODUCTCODE|        CUSTOMERNAME|           PHONE|        ADDRESSLINE1|ADDRESSLINE2|         CITY|STATE|POSTALCODE|COUNTRY|TERRITORY|CONTACTLASTNAME|CONTACTFIRSTNAME|DEALSIZE|
+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+-----+----------+-------+---------+---------------+----------------+--------+
|      10107|             30|     95.7|              2| 2871.0| 2/24/2003 0:00|Shipped| 

In [18]:
sales.createOrReplaceTempView('sales')

In [19]:
data = spark.sql(
                    '''
                    select
                      *
                    from
                          sales
                    '''
)
data.show()

+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+--------+----------+---------+---------+---------------+----------------+--------+
|ORDERNUMBER|QUANTITYORDERED|PRICEEACH|ORDERLINENUMBER|  SALES|      ORDERDATE| STATUS|QTR_ID|MONTH_ID|YEAR_ID|PRODUCTLINE|MSRP|PRODUCTCODE|        CUSTOMERNAME|           PHONE|        ADDRESSLINE1|ADDRESSLINE2|         CITY|   STATE|POSTALCODE|  COUNTRY|TERRITORY|CONTACTLASTNAME|CONTACTFIRSTNAME|DEALSIZE|
+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+--------+----------+---------+---------+---------------+----------------+--------+
|      10107|             30|     95.7|              2| 2871.0| 2/24/2003

In [20]:
calendar = spark.sql(
                    '''
                    select distinct
                      ORDERDATE,
                      QTR_ID,
                      MONTH_ID,
                      YEAR_ID
                    from
                        sales
                    '''
)
calendar.createOrReplaceTempView("calendar")

sales_data = spark.sql(
                    '''
                    select distinct
                      ORDERDATE,
                      SALES,
                      PRICEEACH,
                      ORDERLINENUMBER,
                      PHONE
                    from
                        sales
                    '''
)

sales_data.createOrReplaceTempView("sales_data")

costumer = spark.sql(
                    '''
                    select distinct
                      PHONE,
                      ADDRESSLINE1,
                      ADDRESSLINE2,
                      CITY,
                      STATE,
                      COUNTRY,
                      POSTALCODE
                    from
                        sales
                    '''
)

costumer.createOrReplaceTempView("costumer")

In [21]:
master = spark.sql('''
                    SELECT
                      a.ORDERDATE,
                      a.SALES,
                      a.PRICEEACH,
                      a.ORDERLINENUMBER,
                      a.PHONE,
                      b.ADDRESSLINE1,
                      b.ADDRESSLINE2,
                      b.CITY,
                      b.STATE,
                      b.COUNTRY,
                      b.POSTALCODE
                    FROM
                      sales_data a
                    INNER JOIN
                      costumer b
                    ON
                      a.PHONE = b.PHONE
                  '''
)

In [22]:
master.show()

+---------------+-------+---------+---------------+--------------+--------------------+------------+-------------+--------+---------+----------+
|      ORDERDATE|  SALES|PRICEEACH|ORDERLINENUMBER|         PHONE|        ADDRESSLINE1|ADDRESSLINE2|         CITY|   STATE|  COUNTRY|POSTALCODE|
+---------------+-------+---------+---------------+--------------+--------------------+------------+-------------+--------+---------+----------+
| 11/4/2004 0:00| 6000.4|    100.0|              1|    6035558647|2304 Long Airport...|        NULL|       Nashua|      NH|      USA|     62005|
|  9/3/2004 0:00|1345.68|    56.07|              1| +47 2267 3215|Drammen 121, PR 7...|        NULL|       Bergen|    NULL|   Norway|    N 5804|
| 12/1/2004 0:00|5223.48|    100.0|              7|    2125557413|   4092 Furth Circle|   Suite 400|          NYC|      NY|      USA|     10022|
| 5/20/2003 0:00| 5161.2|    100.0|              3|    40.32.2555|      54, rue Royale|        NULL|       Nantes|    NULL|   Fran